<a href="https://colab.research.google.com/github/vallari24/zero-to-research/blob/main/notebooks/07_gpt_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 GPT From Scratch

TinyStories setup for building GPT from scratch.

In [1]:
# We always start with a dataset to train on. Let's download the TinyStories toy dataset
!wget -O input.txt https://raw.githubusercontent.com/vallari24/zero-to-research/main/data/tiny-input.txt

--2026-06-12 19:19:45--  https://raw.githubusercontent.com/vallari24/zero-to-research/main/data/tiny-input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-06-12 19:19:45 (22.0 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1114838


In [4]:
print(text[:2000])

 Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."
After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.
<|endoftext|>
Once upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy loved to climb. She climbed trees, rocks, and hills. One day, Roxy found an icy hill. She had never seen anything like it before. It was shiny and cold, and she wanted to climb it.
Roxy tried to climb the icy hill, but it was very slippery. She tried again and again, but she kept falling down. Roxy was sad. She wanted to climb the icy hill so much. Then, she saw a little bird named Billy. Billy saw that Roxy was sad and asked, "Why are you sad, Roxy?"
Roxy told Billy about the icy hill and how she couldn't climb it. Billy said, "I have an idea! Let's fi

In [5]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
ids = enc.encode("hello world")
decode = enc.decode(ids)

In [6]:
ids, decode

([31373, 995], 'hello world')

In [7]:
!pip install sentencepiece

import sentencepiece as spm
spm.SentencePieceTrainer.train(
      input="input.txt",
      model_prefix="tokenizer",
      vocab_size=1000,
      model_type="bpe",
      character_coverage=1.0,
  )

In [8]:
sp = spm.SentencePieceProcessor()
sp.load("tokenizer.model")
ids = sp.encode("hello world")
decode = sp.decode(ids)

In [9]:
ids, decode

([21, 51, 924, 304, 66], 'hello world')

In [10]:
# If you want pretrained SentencePiece-style tokenizers, you usually get them from model repos, for example through Hugging Face:
# from transformers import AutoTokenizer
# tok = AutoTokenizer.from_pretrained("google/mt5-small")
# ids = tok.encode("hello world")
# print(ids)
# print(tok.decode(ids))


In [11]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"',-./012345789:;<>?ABCDEFGHIJKLMNOPQRSTUVWYZabcdefghijklmnopqrstuvwxyz| –—‘’“”…
83


In [12]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hello world"))
print(decode(encode("hello world")))

[55, 52, 59, 59, 62, 1, 70, 62, 65, 59, 51]
hello world


In [13]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1114838]) torch.int64
tensor([ 1, 41, 63, 62, 67,  7,  1, 41, 63, 62, 67,  1, 66, 48, 70,  1, 67, 55,
        52,  1, 66, 55, 56, 61, 72,  1, 50, 48, 65,  1, 48, 61, 51,  1, 66, 48,
        56, 51,  5,  1,  3, 45, 62, 70,  5,  1, 33, 56, 67, 67, 72,  5,  1, 72,
        62, 68, 65,  1, 50, 48, 65,  1, 56, 66,  1, 66, 62,  1, 49, 65, 56, 54,
        55, 67,  1, 48, 61, 51,  1, 50, 59, 52, 48, 61,  2,  3,  1, 33, 56, 67,
        67, 72,  1, 66, 60, 56, 59, 52, 51,  1, 48, 61, 51,  1, 65, 52, 63, 59,
        56, 52, 51,  5,  1,  3, 42, 55, 48, 61, 58,  1, 72, 62, 68,  5,  1, 41,
        63, 62, 67,  7,  1, 31,  1, 63, 62, 59, 56, 66, 55,  1, 56, 67,  1, 52,
        69, 52, 65, 72,  1, 51, 48, 72,  7,  3,  0, 23, 53, 67, 52, 65,  1, 63,
        59, 48, 72, 56, 61, 54,  1, 70, 56, 67, 55,  1, 67, 55, 52,  1, 50, 48,
        65,  5,  1, 33, 56, 67, 67, 72,  1, 48, 61, 51,  1, 41, 63, 62, 67,  1,
        53, 52, 59, 67,  1, 67, 55, 56, 65, 66, 67, 72,  7,  1, 42, 55, 52, 72,
      

In [14]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

print(len(data), len(train_data), len(val_data))

1114838 1003354 111484


In [15]:
block_size = 8
train_data[:block_size+1]

tensor([ 1, 41, 63, 62, 67,  7,  1, 41, 63])

In [17]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
  context = x[:t+1]
  label = y[t]
  print(f"when input is {context} the target: {label}")

when input is tensor([1]) the target: 41
when input is tensor([ 1, 41]) the target: 63
when input is tensor([ 1, 41, 63]) the target: 62
when input is tensor([ 1, 41, 63, 62]) the target: 67
when input is tensor([ 1, 41, 63, 62, 67]) the target: 7
when input is tensor([ 1, 41, 63, 62, 67,  7]) the target: 1
when input is tensor([ 1, 41, 63, 62, 67,  7,  1]) the target: 41
when input is tensor([ 1, 41, 63, 62, 67,  7,  1, 41]) the target: 63


In [33]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

ix = torch.randint(len(train_data) - block_size, (batch_size,))
x = torch.stack([train_data[i:i+block_size] for i in ix])
y = torch.stack([train_data[i+1:i+block_size+1] for i in ix])


In [37]:
ix, ix.shape, x, x.shape, y, y.shape

(tensor([636549, 429903, 270558,  12140]),
 torch.Size([4]),
 tensor([[62, 61,  4, 67,  1, 70, 48, 66],
         [70,  1, 66, 55, 56, 65, 67,  1],
         [ 1, 67, 55, 48, 67,  1, 66, 62],
         [66,  1, 69, 52, 65, 72,  1, 55]]),
 torch.Size([4, 8]),
 tensor([[61,  4, 67,  1, 70, 48, 66, 55],
         [ 1, 66, 55, 56, 65, 67,  1, 48],
         [67, 55, 48, 67,  1, 66, 62, 60],
         [ 1, 69, 52, 65, 72,  1, 55, 48]]),
 torch.Size([4, 8]))

In [36]:
for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = x[b, :t+1]
        target = y[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

when input is [62] the target: 61
when input is [62, 61] the target: 4
when input is [62, 61, 4] the target: 67
when input is [62, 61, 4, 67] the target: 1
when input is [62, 61, 4, 67, 1] the target: 70
when input is [62, 61, 4, 67, 1, 70] the target: 48
when input is [62, 61, 4, 67, 1, 70, 48] the target: 66
when input is [62, 61, 4, 67, 1, 70, 48, 66] the target: 55
when input is [70] the target: 1
when input is [70, 1] the target: 66
when input is [70, 1, 66] the target: 55
when input is [70, 1, 66, 55] the target: 56
when input is [70, 1, 66, 55, 56] the target: 65
when input is [70, 1, 66, 55, 56, 65] the target: 67
when input is [70, 1, 66, 55, 56, 65, 67] the target: 1
when input is [70, 1, 66, 55, 56, 65, 67, 1] the target: 48
when input is [1] the target: 67
when input is [1, 67] the target: 55
when input is [1, 67, 55] the target: 48
when input is [1, 67, 55, 48] the target: 67
when input is [1, 67, 55, 48, 67] the target: 1
when input is [1, 67, 55, 48, 67, 1] the target: 6

In [38]:
x

tensor([[62, 61,  4, 67,  1, 70, 48, 66],
        [70,  1, 66, 55, 56, 65, 67,  1],
        [ 1, 67, 55, 48, 67,  1, 66, 62],
        [66,  1, 69, 52, 65, 72,  1, 55]])